# Biotech/Pharma Stock Volatility

I read recently, in the book *Inside the Black Box* by Rishi Narang, that quantitative 
researchers typically are not interested in biotechnology stocks. The specific
passage was: 

>  Third, quants tend to prefer instruments that behave in a
manner conducive to being predicted by systematic models. Returning to
the example of biotechnology stocks, some quants exclude them because
they are subject to sudden, violent price changes based on events such as
government approval or rejection of their latest drug.

The goal of this project is to quantify and visualise whether or not this is actually 
true

In [2]:
import yfinance as yf
import pandas as pd
import plotly.graph_objects as go

## Choice of Stocks

Our goal is to illustrate sudden prices moves, primarily from approval/rejection events.
Therefore, it would be useful to focus on candidates that: 
- are small, because larger companies (typically) have a more diversified set of pipelines/projects,
- have a history of these sources (just easier to identify this way). 

Note/disclaimer: for the second point I did use an LLM to help me generate some 
tickers, as I am not particularly familiar with the history beyond some obvious
ones (e.g. BioNTech with the COVID-19 vaccine). Obviously it would be more thorough
to go through primary sources released by e.g. the FDA, but this is a side project 
so rigour is important but not paramount. :^)

In [ ]:
TICKERS = {
    # "boring" big pharma contrast
    "PFE": "Pfizer",
    # larger company, but still saw big swings during the pandemic
    "MRNA": "Moderna",
    # for the most part stable, as it is a big pharma company, but has Alzeimer edge case
    "LLY": "Eli Lilly",
    # COVID
    "BNTX": "BioNTech",
    # DMD gene therapy
    "SRPT": "Sarepta",
    # Aduhelm Alzeimer's controversy 2023
    "BIIB": "Biogen",
    # late covid vaccine, extreme binary event
    "NVAX": "Novavax",
    # resmetiron NASH approval 2024
    "MDGL": "Madrigal Pharmaceuticals",
    # ETF baseline: is even the basket volatile?
    "XBI": "SPDR S&P Biotech ETF",

    # "REGN": "Regeneron",
    # "VRTX": "Vertex",
    # "ALNY": "Alnylam",
}

## Plotting Stock Prices

Let's start by simply visualising the stock prices for each of these chosen companies.

In [4]:
def fetch_prices(tickers: list[str], start: str, end: str) -> pd.DataFrame:
    """Use yfinance IPA to fetch the daily-adjusted close prices for `tickers`
    between the dates [start, end)

    Return: Pandas DataFrame indexed by date, one column per ticker
    """
    raw = yf.download(
        tickers, start=start, end=end, auto_adjust=True, progress=False, group_by='column'
    )
    if raw.empty:
        raise ValueError(f"yfinance returned empty data for {tickers} for {start} to {end}")

    close = raw["Close"]
    if isinstance(close, pd.Series):
        close = close.to_frame(name=tickers[0])
    return close.dropna(how="all")

DATE_START = "2018-01-01" # pre-pandemic baseline
DATE_END = "2026-06-01" # roughly current date

In [9]:
# making sure it works as expected...
prices = fetch_prices(list(TICKERS.keys()), DATE_START, DATE_END)
prices

$SAGE: possibly delisted; no timezone found

1 Failed download:
['SAGE']: possibly delisted; no timezone found


Ticker,BIIB,BNTX,LLY,MDGL,MRNA,PFE,SAGE,SRPT,XBI
Date,,,,,,,,,
2018-01-02,334.170013,NaN,74.897781,97.419998,NaN,23.443441,NaN,58.250000,86.262939
2018-01-03,339.850006,NaN,75.304642,98.910004,NaN,23.617146,NaN,58.549999,87.410324
2018-01-04,339.989990,NaN,75.640717,97.879997,NaN,23.668613,NaN,57.810001,86.401413
2018-01-05,342.489990,NaN,76.569443,96.529999,NaN,23.713644,NaN,54.869999,86.094780
2018-01-08,329.649994,NaN,76.180283,93.349998,NaN,23.449879,NaN,54.020000,84.383545
...,...,...,...,...,...,...,...,...,...
2026-05-22,193.759995,92.139999,1065.000000,517.260010,46.880001,25.900000,NaN,16.799999,131.530884
2026-05-26,193.080002,92.250000,1064.739990,524.280029,47.029999,25.850000,NaN,16.670000,133.219238
2026-05-27,196.970001,93.000000,1082.920044,527.669983,47.610001,26.209999,NaN,16.680000,134.318146
